# Phase 11 — Uncertainty in PHM

## Why a prediction is not the whole answer

A point prediction gives one value, but a PHM decision also needs to know how uncertain that value is. An estimate near a maintenance threshold with a wide uncertainty range may call for a different action than the same estimate with a narrow, well-calibrated range.

**Readiness warning:** this project does not yet have a defensible measured RUL target, as established in Phase 8. We will not produce a numeric RUL estimate or RUL interval. The calculation below uses a small, explicitly synthetic regression example to teach interval concepts; it is not based on IGBT measurements and is not a PHM result.

## 1. Point prediction, confidence interval, and prediction interval

- **Point prediction:** one estimated value for a quantity of interest, such as a model's predicted mean feature at time `t` or a predicted RUL. It contains no interval or uncertainty statement by itself.
- **Confidence interval (CI):** in the frequentist interpretation, a procedure for estimating an unknown population/model quantity (for example, the conditional mean response). A 95% CI procedure would cover the true quantity in about 95% of repeated samples under its assumptions. It is **not** generally correct to say there is a 95% probability that this fixed true parameter lies in the one realized interval.
- **Prediction interval (PI):** a range intended to cover a **new individual observation** at specified inputs. It must include uncertainty about the mean/model and the new observation's residual variability, so it is wider than a confidence interval for the mean (under the same model and confidence level).

A CI or PI is only as trustworthy as its assumptions, sampling design, and calibration. For time-series data, autocorrelation and device clustering invalidate naive independent-row formulas unless the uncertainty method accounts for that structure. A narrow interval from the wrong model is not reassuring.

## 2. Aleatoric versus epistemic uncertainty

- **Aleatoric uncertainty** is variability/noise inherent in observations or the process at the modeled resolution: sensor noise, repeatability, run-to-run variation, or stochastic future behavior. It is not necessarily removable by collecting more of the same type of data, although better sensors or richer predictors can explain some apparent noise.
- **Epistemic uncertainty** is uncertainty from limited knowledge: too few independent devices, uncertain model form/parameters, missing operating conditions, calibration doubts, or extrapolation beyond observed data. It can often be reduced with relevant additional evidence, but not by repeatedly sampling the same confounded cases alone.

They are not competing labels for every source: the same observed scatter can mix both. In PHM, instrument noise can be aleatoric, while uncertainty about a documented current drift correction is epistemic. Four IGBT devices make cross-device/generalization uncertainty large; the unknown timestamp encoding and undefined cohort-wide EoL are structural uncertainties that a neural-network interval cannot fix.

## 3. Synthetic demonstration: a CI for the mean versus a PI for one observation

We simulate a straight-line relationship with random observation noise. The estimated mean-response CI reflects uncertainty in the fitted line; the individual-observation PI also includes residual noise. The simulation is pedagogical: its parameters and data are invented, it uses independent Gaussian errors, and its intervals do not describe IGBT voltage, current, device health, or RUL.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import t

rng = np.random.default_rng(2026)
x = np.linspace(0.0, 10.0, 30)
true_intercept, true_slope, noise_sd = 2.0, 0.8, 1.4
y = true_intercept + true_slope * x + rng.normal(0.0, noise_sd, size=x.size)

design = np.column_stack([np.ones_like(x), x])
beta, _, _, _ = np.linalg.lstsq(design, y, rcond=None)
fitted = design @ beta
degrees_freedom = len(x) - design.shape[1]
residual_sd = np.sqrt(np.sum((y - fitted) ** 2) / degrees_freedom)
covariance_beta = residual_sd**2 * np.linalg.inv(design.T @ design)

x_grid = np.linspace(x.min(), x.max(), 200)
grid_design = np.column_stack([np.ones_like(x_grid), x_grid])
mean_prediction = grid_design @ beta
mean_se = np.sqrt(np.einsum("ij,jk,ik->i", grid_design, covariance_beta, grid_design))
critical_value = t.ppf(0.975, degrees_freedom)
mean_ci_low = mean_prediction - critical_value * mean_se
mean_ci_high = mean_prediction + critical_value * mean_se
observation_se = np.sqrt(mean_se**2 + residual_sd**2)
prediction_low = mean_prediction - critical_value * observation_se
prediction_high = mean_prediction + critical_value * observation_se

fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
ax.scatter(x, y, s=28, alpha=0.75, label="synthetic observations")
ax.plot(x_grid, mean_prediction, color="black", label="point prediction (fitted mean)")
ax.fill_between(x_grid, mean_ci_low, mean_ci_high, color="tab:blue", alpha=0.28,
                label="95% CI for mean response")
ax.fill_between(x_grid, prediction_low, prediction_high, color="tab:orange", alpha=0.18,
                label="95% PI for one new observation")
ax.set(title="Synthetic illustration only — not IGBT data or RUL",
       xlabel="Illustrative input", ylabel="Illustrative response")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

print(f"Estimated residual standard deviation (synthetic): {residual_sd:.3f}")
print("At each x, the prediction interval includes residual variation and is wider than the mean CI.")

The CI widens near the edges because the fitted mean is less certain away from the center of the observed inputs. The PI is wider throughout because a future individual response also has observation-level noise. Both intervals rely on the assumed linear model, independent errors, and approximately Gaussian residuals; a time series with correlated measurements needs a different uncertainty calculation.

## 4. What uncertainty means for this IGBT dataset

If a future study estimates a candidate feature, possible **aleatoric** contributions include capture-to-capture measurement noise, repeatability, and real short-term variation under nominally identical conditions. Possible **epistemic** contributions include only four devices, unknown calibration/collector-current drift, missing or uneven captures, changing stress conditions, uncertain relationship between recorded clock and accumulated aging, competing degradation mechanisms, and uncertainty in EoL/event-time alignment.

For a future RUL model, one might conceptually decompose predictive variance as:

$$\mathrm{Var}(Y \mid x, D) = \underbrace{\mathbb{E}_{\theta}[\mathrm{Var}(Y \mid x,\theta)]}_{\text{aleatoric}} + \underbrace{\mathrm{Var}_{\theta}(\mathbb{E}[Y \mid x,\theta])}_{\text{epistemic}}$$

where `D` is the training evidence and `theta` represents uncertain model parameters. This decomposition is model-dependent; neither term is currently quantifiable as RUL uncertainty because there is no justified RUL label/time axis for this cohort. Do not attach a numerical RUL confidence band to the existing remaining-observation-horizon proxy and call it failure-life uncertainty.

## 5. Later approaches for model uncertainty

- **Monte Carlo Dropout:** keep dropout active for repeated forward passes at prediction time. The spread approximates uncertainty over model functions under specific Bayesian interpretations/assumptions. It is not automatically calibrated and can miss distribution shift or label uncertainty.
- **Ensemble models:** train several models (ideally with resampling or different initializations) and use prediction disagreement as an epistemic-uncertainty proxy. They are straightforward and useful, but shared architecture/data bias can make all ensemble members confidently wrong. Use device/run-aware resampling, not random capture-row bootstrap.
- **Probabilistic models:** predict a distribution or parameters (e.g., mean and variance) rather than one value, or use survival/time-to-event likelihoods that represent censoring. These can model heteroscedastic aleatoric variation, but the likelihood, censoring mechanism, and event target must be correctly specified.

A useful uncertainty method needs evaluation too: check empirical coverage of nominal 90%/95% intervals on untouched, deployment-matched devices/runs; inspect interval width and conditional coverage across devices, stress settings, and prediction horizons; and test under distribution shift. Good coverage with unusably wide intervals is not automatically useful, and average coverage can hide a poorly covered subgroup.

## Phase 11 learning check

1. Which uncertainty interval concerns an estimated mean response, and which aims to cover one new observation?
2. Why is a prediction interval wider than a confidence interval for the mean in the synthetic example?
3. Give one PHM example each of aleatoric and epistemic uncertainty in this IGBT dataset.
4. Why can repeated Monte Carlo Dropout passes not repair an invalid EoL definition?
5. How would you check whether 95% intervals are calibrated on future devices?
6. Why is no numerical RUL interval reported here?

**Readiness decision:** intervals quantify uncertainty conditional on a target, data, and model assumptions. Resolve the Phase 8 RUL definition and improve independent validation evidence before quantifying uncertainty in RUL.